# 03 - Clustering models and segments

K-Means, hierarchical, DBSCAN, stability, the choice of k and the final segments (Phases 4 and 5).

This notebook reads the outputs in `results/` (produced by the scripts in `src/`); it does not repeat the cleaning logic.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
RES = ROOT / "results"
pd.set_option("display.width", 200)

def csv(name):
    path = RES / name
    if not path.exists():
        print(f"{name} not found: run `python src/run_all.py` from the repository root first.")
        return None
    return pd.read_csv(path)

def js(name):
    path = RES / name
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


## K-Means, k = 2-10

In [ ]:
km = csv('kmeans_metrics.csv'); display(km)
if km is not None:
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
    ax[0].plot(km.k, km.inertia, marker='o'); ax[0].set_title('Elbow (inertia)')
    col = 'silhouette_full' if 'silhouette_full' in km else 'silhouette_sampled'
    ax[1].plot(km.k, km[col], marker='o'); ax[1].set_title(f'Silhouette ({col})')
    ax[2].plot(km.k, km.davies_bouldin, marker='o'); ax[2].set_title('Davies-Bouldin (lower is better)')
    plt.tight_layout(); plt.show()

## Stability and hierarchical clustering

In [ ]:
display(csv('kmeans_stability.csv'))
h = csv('hierarchical_metrics.csv')
if h is not None:
    display(h.pivot(index='k', columns='linkage', values='silhouette').round(3))
    display(h.groupby('linkage').cophenetic_correlation.first())

## DBSCAN

In [ ]:
print(js('clustering_comparison.json').get('dbscan', {}).get('finding'))
g = csv('dbscan_grid.csv'); display(g.sort_values('silhouette_non_noise', ascending=False).head(10))
display(csv('dbscan_noise_profile.csv'))

## Algorithm comparison (k = 3-6)

In [ ]:
display(csv('algorithm_comparison.csv'))

## Choice of k and final segments

In [ ]:
print(js('segment_summary.json').get('selection_rule'))
display(csv('k_selection.csv'))
prof = csv('segment_profiles.csv')
display(prof.drop(columns=['description','recommended_actions']))

In [ ]:
if prof is not None:
    prof.set_index('SegmentName')[['customer_share','revenue_share']].plot(kind='bar', figsize=(8,4)); plt.ylabel('share'); plt.tight_layout(); plt.show()
    for r in prof.itertuples(): print(f'- {r.SegmentName}: {r.recommended_actions}')